In [ ]:
from pathlib import Path
import json
import time
import random
import inspect

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

current_directory = Path.cwd().resolve()
if (current_directory / 'final-2' / 'output').exists():
    project_folder = current_directory / 'final-2'
elif current_directory.name == 'notebooks' and (current_directory.parent / 'output').exists():
    project_folder = current_directory.parent
else:
    project_folder = current_directory

dataset_path = project_folder / 'output' / 'particle_ugradu_dataset.npz'
results_folder = project_folder / 'output' / 'task1_v3_simple_training'
results_folder.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Project folder:', project_folder)
print('Dataset path  :', dataset_path)
print('Results folder:', results_folder)
print('Device        :', device)
if device.type == 'cuda':
    print('GPU           :', torch.cuda.get_device_name(0))

if not dataset_path.exists():
    raise FileNotFoundError('Missing particle_ugradu_dataset.npz. Run final-2/preprocess_data.py first.')


In [ ]:
# Load the preprocessed particle frames.
# The preprocessing file stores raw arrays plus normalized arrays.
# Normalization is computed from training frames only, then applied to every split.

dataset_file = np.load(dataset_path, allow_pickle=True)

feature_names = [str(name) for name in dataset_file['feature_names'].tolist()]
target_names = [str(name) for name in dataset_file['target_names'].tolist()]
frame_contexts = list(dataset_file['frame_contexts'])
frame_ranges = list(dataset_file['frame_ranges'])

In [ ]:
if 'inputs_by_frame_norm' not in dataset_file or 'targets_by_frame_norm' not in dataset_file:
    raise RuntimeError('This notebook expects frame-wise arrays from preprocess_data.py.')

inputs_by_frame_normalized = [np.asarray(x, dtype=np.float32) for x in dataset_file['inputs_by_frame_norm'].tolist()]
targets_by_frame_normalized = [np.asarray(y, dtype=np.float32) for y in dataset_file['targets_by_frame_norm'].tolist()]
inputs_by_frame_raw = [np.asarray(x, dtype=np.float32) for x in dataset_file['inputs_by_frame'].tolist()]
targets_by_frame_raw = [np.asarray(y, dtype=np.float32) for y in dataset_file['targets_by_frame'].tolist()]

# Plain split names used throughout this notebook.
# The preprocessing script assigns cases from folder names and saves frame IDs here.
training_frame_ids = dataset_file['train_frame_ids'].astype(np.int64)
# Primary validation is same-distribution validation from the training cases.
# Held-out validation-angle cases are kept separately for interpolation diagnostics.
validation_frame_ids = dataset_file['val_id_frame_ids'].astype(np.int64) if 'val_id_frame_ids' in dataset_file.files else dataset_file['val_frame_ids'].astype(np.int64)
validation_angle_frame_ids = dataset_file['validation_angle_frame_ids'].astype(np.int64) if 'validation_angle_frame_ids' in dataset_file.files else np.array([], dtype=np.int64)
testing_frame_ids = dataset_file['test_frame_ids'].astype(np.int64) if 'test_frame_ids' in dataset_file.files else np.array([], dtype=np.int64)


def frame_context_as_dict(frame_id):
    # np.savez stores dictionaries as object arrays; this helper converts them back safely.
    context = frame_contexts[int(frame_id)]
    if isinstance(context, dict):
        return context
    if hasattr(context, 'item'):
        maybe_dict = context.item()
        if isinstance(maybe_dict, dict):
            return maybe_dict
    return dict(context)


def test_frame_ids_for_role(role_name, saved_key, aoa_fallback=None):
    # Prefer explicit preprocessing keys. Fallback keeps older preprocessed files readable.
    if saved_key in dataset_file.files:
        return dataset_file[saved_key].astype(np.int64)
    selected = []
    for frame_id in testing_frame_ids:
        context = frame_context_as_dict(int(frame_id))
        if str(context.get('test_role', '')) == role_name:
            selected.append(int(frame_id))
            continue
        if aoa_fallback is not None and int(round(float(context.get('aoa_deg', -999)))) == int(aoa_fallback):
            selected.append(int(frame_id))
    return np.asarray(selected, dtype=np.int64)


# The testing split intentionally contains three different questions:
# 1. normal testing: AoA 27, same particle shedding rate as training
# 2. super-resolution testing: AoA 19, two particles shed per step
# 3. unseen-angle testing: AoA 32, outside the main training/validation AoA plan
testing_normal_frame_ids = test_frame_ids_for_role('testing_normal', 'test_normal_frame_ids', aoa_fallback=27)
testing_super_resolution_frame_ids = test_frame_ids_for_role(
    'testing_super_resolution', 'test_super_resolution_frame_ids', aoa_fallback=19
)
testing_unseen_angle_frame_ids = test_frame_ids_for_role('testing_unseen_angle', 'test_unseen_angle_frame_ids', aoa_fallback=32)

has_validation_data = len(validation_frame_ids) > 0
has_testing_data = len(testing_frame_ids) > 0

output_mean = torch.tensor(dataset_file['out_mean'].astype(np.float32), device=device)
output_standard_deviation = torch.tensor(dataset_file['out_std'].astype(np.float32), device=device)

input_dimension = int(inputs_by_frame_normalized[0].shape[1])
output_dimension = int(targets_by_frame_normalized[0].shape[1])

In [ ]:
print('Number of frames      :', len(inputs_by_frame_normalized))
print('Input feature count   :', input_dimension)
print('Output target count   :', output_dimension)
print('Training frames       :', len(training_frame_ids))
print('Validation frames     :', len(validation_frame_ids), '(same-distribution frames from training cases)')
print('Held-out angle validation frames:', len(validation_angle_frame_ids))
print('Testing frames        :', len(testing_frame_ids))
print('  normal test frames  :', len(testing_normal_frame_ids))
print('  super-res test frames:', len(testing_super_resolution_frame_ids))
print('  unseen-angle frames :', len(testing_unseen_angle_frame_ids))
print('Input feature names   :', feature_names)
print('Output target names   :', target_names)
if not has_validation_data:
    print('[info] No validation frames yet. The notebook will skip validation plots/metrics until those cases exist.')
if not has_testing_data:
    print('[info] No testing frames yet. The notebook will skip testing plots/metrics until those cases exist.')


In [ ]:
# Data integrity checks.
# These checks catch common silent failures before model training: missing values, wrong shapes, and split leakage.

def check_frame_arrays(frame_ids, split_name, maximum_frames_to_check=50):
    if len(frame_ids) == 0:
        print(f'[{split_name}] no frames available yet; skipping array checks.')
        return

    for frame_id in np.asarray(frame_ids[:maximum_frames_to_check], dtype=np.int64):
        input_raw = inputs_by_frame_raw[int(frame_id)]
        target_raw = targets_by_frame_raw[int(frame_id)]
        input_normalized = inputs_by_frame_normalized[int(frame_id)]
        target_normalized = targets_by_frame_normalized[int(frame_id)]

        if input_raw.ndim != 2 or input_raw.shape[1] != input_dimension:
            raise RuntimeError(f'{split_name} frame {frame_id}: bad input shape {input_raw.shape}')
        if target_raw.ndim != 2 or target_raw.shape[1] != output_dimension:
            raise RuntimeError(f'{split_name} frame {frame_id}: bad target shape {target_raw.shape}')
        if input_raw.shape[0] != target_raw.shape[0]:
            raise RuntimeError(f'{split_name} frame {frame_id}: input/target particle count mismatch')
        if not np.isfinite(input_raw).all() or not np.isfinite(input_normalized).all():
            raise RuntimeError(f'{split_name} frame {frame_id}: NaN or inf in input')
        if not np.isfinite(target_raw).all() or not np.isfinite(target_normalized).all():
            raise RuntimeError(f'{split_name} frame {frame_id}: NaN or inf in target')

    print(f'[{split_name}] checked {min(len(frame_ids), maximum_frames_to_check)} frames successfully.')


def cases_for_frame_ids(frame_ids):
    cases = []
    for frame_id in np.asarray(frame_ids, dtype=np.int64):
        cases.append(str(frame_ranges[int(frame_id)][0]))
    return set(cases)

check_frame_arrays(training_frame_ids, 'training')
check_frame_arrays(validation_frame_ids, 'validation')
check_frame_arrays(testing_frame_ids, 'testing')

training_cases = cases_for_frame_ids(training_frame_ids)
validation_cases = cases_for_frame_ids(validation_frame_ids)
testing_cases = cases_for_frame_ids(testing_frame_ids)

print('Training cases  :', sorted(training_cases))
print('Validation cases:', sorted(validation_cases))
print('Testing cases   :', sorted(testing_cases))

# Same-distribution validation is intentionally drawn from the training cases.
# It is used for tuning/early stopping, not for final generalization claims.
shared_train_validation_cases = training_cases & validation_cases
print('Shared train/validation cases:', sorted(shared_train_validation_cases))

# Testing must remain case-disjoint from anything used for fitting or tuning.
if training_cases & testing_cases:
    raise RuntimeError('Case leakage: at least one case appears in both training and testing.')
if validation_cases & testing_cases:
    raise RuntimeError('Case leakage: at least one case appears in both validation and testing.')


In [ ]:
# Quick data scale check.
# This tells us whether validation/testing target magnitudes live in the same range as training.
# Large differences here usually mean generalization will be hard even if the code is correct.

def sample_target_magnitudes(frame_ids, maximum_frames=40, maximum_particles_per_frame=12000):
    if len(frame_ids) == 0:
        return None

    chosen_frame_ids = np.asarray(frame_ids[:maximum_frames], dtype=np.int64)
    velocity_magnitudes = []
    gradient_magnitudes = []

    for frame_id in chosen_frame_ids:
        target = targets_by_frame_raw[int(frame_id)]
        if target.shape[0] > maximum_particles_per_frame:
            picked = np.random.default_rng(SEED + int(frame_id)).choice(
                target.shape[0], size=maximum_particles_per_frame, replace=False
            )
            target = target[picked]

        velocity_magnitudes.append(np.linalg.norm(target[:, :3], axis=1))
        gradient_magnitudes.append(np.linalg.norm(target[:, 3:], axis=1))

    velocity_magnitudes = np.concatenate(velocity_magnitudes)
    gradient_magnitudes = np.concatenate(gradient_magnitudes)
    return {
        'mean_velocity': float(np.mean(velocity_magnitudes)),
        'median_velocity': float(np.median(velocity_magnitudes)),
        'mean_velocity_gradient': float(np.mean(gradient_magnitudes)),
        'median_velocity_gradient': float(np.median(gradient_magnitudes)),
        'velocity_95_percentile': float(np.quantile(velocity_magnitudes, 0.95)),
        'velocity_gradient_95_percentile': float(np.quantile(gradient_magnitudes, 0.95)),
    }

split_statistics = {
    'training': sample_target_magnitudes(training_frame_ids),
    'validation': sample_target_magnitudes(validation_frame_ids),
    'testing': sample_target_magnitudes(testing_frame_ids),
}

print(json.dumps(split_statistics, indent=2))

training_velocity_mean = split_statistics['training']['mean_velocity']
for split_name in ['validation', 'testing']:
    if split_statistics[split_name] is None:
        continue
    ratio = split_statistics[split_name]['mean_velocity'] / max(training_velocity_mean, 1e-12)
    print(f'{split_name} mean |u| / training mean |u| = {ratio:.3f}')

print('')
print('Interpretation: target normalization helps optimization, but it cannot fully remove a physics distribution shift.')
print('If testing magnitudes are very different from training magnitudes, add training cases that cover that range.')


In [ ]:
# Normalization audit.
# preprocess_data.py computes mean/std from training particles only, then applies that scaling to every split.
# This cell prints the stored values and checks whether the normalized training data is centered near zero.

input_mean = dataset_file['in_mean'].astype(np.float32)
input_standard_deviation = dataset_file['in_std'].astype(np.float32)
target_mean = dataset_file['out_mean'].astype(np.float32)
target_standard_deviation = dataset_file['out_std'].astype(np.float32)

print('Normalization source: final-2/preprocess_data.py')
print('Relevant lines in preprocessing:')
print('  X_train = concatenate training-frame inputs')
print('  Y_train = concatenate training-frame targets')
print('  in_mean = mean(X_train), in_std = std(X_train)')
print('  out_mean = mean(Y_train), out_std = std(Y_train)')
print('  normalized = (raw - mean) / std')

print('\nFirst input channels:')
for name, mean_value, std_value in zip(feature_names[:min(8, len(feature_names))], input_mean.reshape(-1)[:8], input_standard_deviation.reshape(-1)[:8]):
    print(f'  {name:24s} mean={mean_value: .4e} std={std_value: .4e}')

print('\nOutput channels:')
for name, mean_value, std_value in zip(target_names, target_mean.reshape(-1), target_standard_deviation.reshape(-1)):
    print(f'  {name:24s} mean={mean_value: .4e} std={std_value: .4e}')

# Use a limited number of frames so this audit is fast even for large datasets.
def normalized_channel_summary(frame_ids, arrays_by_frame, maximum_frames=30):
    chosen = frame_ids[:maximum_frames]
    values = np.concatenate([arrays_by_frame[int(frame_id)] for frame_id in chosen], axis=0)
    return np.mean(values, axis=0), np.std(values, axis=0)

training_input_mean_after_scaling, training_input_std_after_scaling = normalized_channel_summary(
    training_frame_ids, inputs_by_frame_normalized
)
training_target_mean_after_scaling, training_target_std_after_scaling = normalized_channel_summary(
    training_frame_ids, targets_by_frame_normalized
)

print('\nTraining split check after normalization, using a frame sample:')
print('  input mean range :', float(training_input_mean_after_scaling.min()), 'to', float(training_input_mean_after_scaling.max()))
print('  input std range  :', float(training_input_std_after_scaling.min()), 'to', float(training_input_std_after_scaling.max()))
print('  target mean range:', float(training_target_mean_after_scaling.min()), 'to', float(training_target_mean_after_scaling.max()))
print('  target std range :', float(training_target_std_after_scaling.min()), 'to', float(training_target_std_after_scaling.max()))


In [ ]:
# Zero/near-zero target-frame diagnostic.
# Relative error is not meaningful if the true field is zero, because it divides by ||target||.
# These frames can still be used for training with MSE/SmoothL1, but they should be skipped in relative-error reporting.

def target_frame_norms(frame_ids):
    rows = []
    for frame_id in np.asarray(frame_ids, dtype=np.int64):
        target = targets_by_frame_raw[int(frame_id)]
        context = frame_context_as_dict(int(frame_id)) if 'frame_context_as_dict' in globals() else frame_contexts[int(frame_id)]
        rows.append({
            'frame_id': int(frame_id),
            'case': str(context.get('case', 'unknown')),
            'frame': str(context.get('frame', 'unknown')),
            'norm': float(np.linalg.norm(target.reshape(-1))),
            'velocity_norm': float(np.linalg.norm(target[:, :3].reshape(-1))),
            'gradient_norm': float(np.linalg.norm(target[:, 3:].reshape(-1))),
        })
    return rows

for split_name, frame_ids in [('training', training_frame_ids), ('validation', validation_frame_ids), ('testing', testing_frame_ids)]:
    rows = target_frame_norms(frame_ids[:40])
    zero_rows = [row for row in rows if row['norm'] <= 1e-12]
    finite_norms = np.asarray([row['norm'] for row in rows], dtype=np.float64)
    if finite_norms.size == 0:
        print(f'{split_name}: no frames')
        continue
    print(
        f"{split_name}: checked first {len(rows)} frames | "
        f"zero-target frames={len(zero_rows)} | "
        f"norm min/median/max={finite_norms.min():.3e}/{np.median(finite_norms):.3e}/{finite_norms.max():.3e}"
    )
    if zero_rows:
        print('  examples:', zero_rows[:3])


In [ ]:
# Plot target magnitude distributions for training, validation, and testing.
# If these distributions are far apart, the model is being asked to extrapolate.

def collect_target_magnitudes_for_plot(frame_ids, maximum_frames=40, maximum_particles_per_frame=8000):
    if len(frame_ids) == 0:
        return None
    velocity_values = []
    gradient_values = []
    rng = np.random.default_rng(SEED)

    for frame_id in np.asarray(frame_ids[:maximum_frames], dtype=np.int64):
        target = targets_by_frame_raw[int(frame_id)]
        if target.shape[0] > maximum_particles_per_frame:
            picked = rng.choice(target.shape[0], size=maximum_particles_per_frame, replace=False)
            target = target[picked]
        velocity_values.append(np.linalg.norm(target[:, :3], axis=1))
        gradient_values.append(np.linalg.norm(target[:, 3:], axis=1))

    return np.concatenate(velocity_values), np.concatenate(gradient_values)

plot_magnitudes = {
    'training': collect_target_magnitudes_for_plot(training_frame_ids),
    'validation': collect_target_magnitudes_for_plot(validation_frame_ids),
    'testing': collect_target_magnitudes_for_plot(testing_frame_ids),
}

figure, axes = plt.subplots(1, 2, figsize=(13.5, 4.5), constrained_layout=True)
colors = {'training': '#4c78a8', 'validation': '#59a14f', 'testing': '#e15759'}

for split_name, values in plot_magnitudes.items():
    if values is None:
        continue
    velocity_values, gradient_values = values
    axes[0].hist(velocity_values, bins=80, density=True, histtype='step', linewidth=2.0, color=colors[split_name], label=split_name)
    axes[1].hist(gradient_values, bins=80, density=True, histtype='step', linewidth=2.0, color=colors[split_name], label=split_name)

axes[0].set_title('Velocity magnitude distribution')
axes[0].set_xlabel('|u|')
axes[0].set_ylabel('density')
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].set_title('Velocity-gradient magnitude distribution')
axes[1].set_xlabel('|gradU|')
axes[1].set_ylabel('density')
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

plot_path = results_folder / 'target_magnitude_distributions.png'
figure.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)


In [ ]:
# Plot important input feature distributions for training, validation, and testing.
# This checks whether the model sees the same kind of particle states in each split.

important_feature_candidates = [
    'x', 'y', 'z', 'Gamma_x', 'Gamma_y', 'Gamma_z', 'Gamma_mag', 'sigma',
    'geom_dist', 'geom_body_near', 'angle_of_attack', 'freestream_x', 'freestream_z',
    'phase', 'local_neighbor_count', 'nearest_particle_distance', 'static_particle_flag',
]
important_features = [name for name in important_feature_candidates if name in feature_names]
if len(important_features) == 0:
    important_features = feature_names[:min(8, len(feature_names))]


def collect_feature_values(frame_ids, feature_name, normalized=False, maximum_frames=30, maximum_particles_per_frame=8000):
    if len(frame_ids) == 0:
        return None
    feature_index = feature_names.index(feature_name)
    values = []
    rng = np.random.default_rng(SEED + feature_index)
    source_arrays = inputs_by_frame_normalized if normalized else inputs_by_frame_raw

    for frame_id in np.asarray(frame_ids[:maximum_frames], dtype=np.int64):
        frame = source_arrays[int(frame_id)]
        if frame.shape[0] > maximum_particles_per_frame:
            picked = rng.choice(frame.shape[0], size=maximum_particles_per_frame, replace=False)
            frame = frame[picked]
        values.append(frame[:, feature_index])

    return np.concatenate(values)

splits_for_plot = {
    'training': training_frame_ids,
    'validation': validation_frame_ids,
    'testing': testing_frame_ids,
}
colors = {'training': '#4c78a8', 'validation': '#59a14f', 'testing': '#e15759'}

columns = 3
rows = int(np.ceil(len(important_features) / columns))
figure, axes = plt.subplots(rows, columns, figsize=(5.0 * columns, 3.4 * rows), constrained_layout=True)
axes = np.asarray(axes).reshape(-1)

for axis, feature_name in zip(axes, important_features):
    for split_name, frame_ids in splits_for_plot.items():
        values = collect_feature_values(frame_ids, feature_name, normalized=False)
        if values is None:
            continue
        axis.hist(values, bins=60, density=True, histtype='step', linewidth=1.8, color=colors[split_name], label=split_name)
    axis.set_title(feature_name)
    axis.grid(alpha=0.25)

for axis in axes[len(important_features):]:
    axis.axis('off')
axes[0].legend(frameon=False)
figure.suptitle('Raw input feature distributions', fontsize=14)
plot_path = results_folder / 'input_feature_distributions_raw.png'
figure.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)


# Data-pair visualization for GNO learning

This section visualizes what one training example actually looks like to the Graph Neural Operator. The model receives an unordered particle cloud with per-particle features, and it predicts per-particle velocity and velocity-gradient targets on the same particle set.

These plots follow common point-cloud/operator-learning diagnostics: show the point cloud geometry, inspect local density/feature distributions, and compare input conditioning against target magnitudes. This is useful because GNO performance depends strongly on particle distribution, local neighborhood coverage, target scale, and whether validation/testing cases live inside the same feature range as training.

Reference ideas: PointNet treats point clouds as unordered geometric sets; PointNet++ emphasizes local metric neighborhoods and density variation; NeuralOperator's GNO documentation describes learning maps between functions evaluated on arbitrary point clouds.


In [ ]:
# Visualize input-output pairs and data distributions in a GNO-friendly way.
# This is not a training plot. It is a dataset sanity/interpretability plot.
# It answers: what does one frame look like, what are the model inputs, and what targets must be learned?

split_frame_ids = {
    'training': training_frame_ids,
    'validation': validation_frame_ids,
    'validation_angle': validation_angle_frame_ids,
    'testing_normal': testing_normal_frame_ids,
    'testing_super_resolution': testing_super_resolution_frame_ids,
    'testing_unseen_angle': testing_unseen_angle_frame_ids,
}

split_colors = {
    'training': '#4c78a8',
    'validation': '#59a14f',
    'validation_angle': '#edc948',
    'testing_normal': '#e15759',
    'testing_super_resolution': '#b07aa1',
    'testing_unseen_angle': '#f28e2b',
}


def feature_index_or_none(name):
    # Return the column index of an input feature if it exists in this dataset.
    return feature_names.index(name) if name in feature_names else None


def target_index_or_none(name):
    # Return the column index of an output target if it exists in this dataset.
    return target_names.index(name) if name in target_names else None


def context_for_frame(frame_id):
    # frame_contexts is saved by preprocess_data.py and keeps case/frame metadata attached to each pair.
    context = frame_contexts[int(frame_id)]
    return context if isinstance(context, dict) else dict(context.item())


def choose_representative_frame(frame_ids, split_name):
    # Pick the middle frame of a split so we do not always visualize the first transient frame.
    if len(frame_ids) == 0:
        return None
    ordered = np.asarray(frame_ids, dtype=np.int64)
    return int(ordered[len(ordered) // 2])


def sample_points_for_plot(input_raw, target_raw, maximum_points=9000, seed_offset=0):
    # Large particle clouds are expensive and visually saturated, so plot a reproducible subset.
    particle_count = input_raw.shape[0]
    if particle_count <= maximum_points:
        return input_raw, target_raw
    rng = np.random.default_rng(SEED + seed_offset)
    chosen = rng.choice(particle_count, size=maximum_points, replace=False)
    return input_raw[chosen], target_raw[chosen]


def robust_color_limits(values, lower=2.0, upper=98.0):
    # Percentile limits prevent one extreme particle from washing out the whole color scale.
    values = np.asarray(values)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return 0.0, 1.0
    low, high = np.percentile(finite, [lower, upper])
    if np.isclose(low, high):
        low, high = float(finite.min()), float(finite.max())
    if np.isclose(low, high):
        high = low + 1.0
    return float(low), float(high)


def add_3d_cloud(axis, coordinates, color_values, title, colorbar_label, cmap='viridis'):
    # Coordinates are always input particle locations; colors show either input features or output targets.
    low, high = robust_color_limits(color_values)
    plot = axis.scatter(
        coordinates[:, 0], coordinates[:, 1], coordinates[:, 2],
        c=color_values, s=2.0, alpha=0.72, cmap=cmap, vmin=low, vmax=high, linewidths=0.0,
    )
    axis.set_title(title)
    axis.set_xlabel('x')
    axis.set_ylabel('y')
    axis.set_zlabel('z')
    axis.view_init(elev=18, azim=-58)
    plt.colorbar(plot, ax=axis, shrink=0.65, pad=0.02, label=colorbar_label)


def plot_input_output_pair(split_name, frame_ids):
    # One pair means: input particle features at a frame, and output u/gradU targets for those particles.
    frame_id = choose_representative_frame(frame_ids, split_name)
    if frame_id is None:
        print(f'[{split_name}] no frames available; skipping pair visualization.')
        return

    input_raw = inputs_by_frame_raw[frame_id]
    target_raw = targets_by_frame_raw[frame_id]
    input_plot, target_plot = sample_points_for_plot(input_raw, target_raw, seed_offset=frame_id)

    x_i, y_i, z_i = feature_index_or_none('x'), feature_index_or_none('y'), feature_index_or_none('z')
    if None in (x_i, y_i, z_i):
        raise RuntimeError('Expected x, y, z columns in input features for point-cloud visualization.')

    coordinates = input_plot[:, [x_i, y_i, z_i]]

    gamma_columns = [feature_index_or_none(name) for name in ['Gamma_x', 'Gamma_y', 'Gamma_z']]
    if all(index is not None for index in gamma_columns):
        circulation_color = np.linalg.norm(input_plot[:, gamma_columns], axis=1)
        circulation_label = '|Gamma| input'
    elif feature_index_or_none('Gamma_mag') is not None:
        circulation_color = input_plot[:, feature_index_or_none('Gamma_mag')]
        circulation_label = 'Gamma_mag input'
    else:
        circulation_color = np.zeros(input_plot.shape[0], dtype=np.float32)
        circulation_label = 'circulation unavailable'

    sigma_i = feature_index_or_none('sigma')
    sigma_color = input_plot[:, sigma_i] if sigma_i is not None else np.zeros(input_plot.shape[0], dtype=np.float32)

    velocity_magnitude = np.linalg.norm(target_plot[:, :3], axis=1)
    gradient_magnitude = np.linalg.norm(target_plot[:, 3:], axis=1)

    context = context_for_frame(frame_id)
    case_name = str(context.get('case', 'unknown case'))
    physical_frame = str(context.get('frame', context.get('fr', 'unknown frame')))
    particle_count = int(context.get('n_particles', input_raw.shape[0]))

    figure = plt.figure(figsize=(15, 11), constrained_layout=True)
    axes = [figure.add_subplot(2, 2, i + 1, projection='3d') for i in range(4)]

    add_3d_cloud(axes[0], coordinates, circulation_color, 'Input cloud colored by circulation', circulation_label, cmap='coolwarm')
    add_3d_cloud(axes[1], coordinates, sigma_color, 'Input cloud colored by core radius', 'sigma input', cmap='magma')
    add_3d_cloud(axes[2], coordinates, velocity_magnitude, 'Output target: velocity magnitude', '|u| target', cmap='turbo')
    add_3d_cloud(axes[3], coordinates, gradient_magnitude, 'Output target: velocity-gradient magnitude', '|gradU| target', cmap='plasma')

    figure.suptitle(
        f'{split_name} input-output pair | case={case_name} | frame={physical_frame} | particles={particle_count:,}',
        fontsize=14,
    )
    plot_path = results_folder / f'{split_name.lower()}_input_output_pair_cloud.png'
    figure.savefig(plot_path, dpi=230, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


for split_name, frame_ids in split_frame_ids.items():
    plot_input_output_pair(split_name, frame_ids)


In [ ]:
# Split-level distribution diagnostics for GNO learning.
# These plots reveal whether validation/testing require interpolation or extrapolation in particle count,
# AoA/phase, local geometry, and target magnitude.


def collect_frame_level_summary(frame_ids, maximum_frames=80, maximum_particles_per_frame=10000):
    rows = []
    rng = np.random.default_rng(SEED + 991)
    for frame_id in np.asarray(frame_ids[:maximum_frames], dtype=np.int64):
        input_raw = inputs_by_frame_raw[int(frame_id)]
        target_raw = targets_by_frame_raw[int(frame_id)]
        if input_raw.shape[0] > maximum_particles_per_frame:
            chosen = rng.choice(input_raw.shape[0], size=maximum_particles_per_frame, replace=False)
            input_used = input_raw[chosen]
            target_used = target_raw[chosen]
        else:
            input_used = input_raw
            target_used = target_raw

        context = context_for_frame(int(frame_id))
        row = {
            'frame_id': int(frame_id),
            'case': str(context.get('case', 'unknown')),
            'physical_frame': float(context.get('frame', context.get('fr', np.nan))),
            'particle_count': int(input_raw.shape[0]),
            'velocity_median': float(np.median(np.linalg.norm(target_used[:, :3], axis=1))),
            'velocity_95': float(np.percentile(np.linalg.norm(target_used[:, :3], axis=1), 95)),
            'gradient_median': float(np.median(np.linalg.norm(target_used[:, 3:], axis=1))),
            'gradient_95': float(np.percentile(np.linalg.norm(target_used[:, 3:], axis=1), 95)),
        }

        for optional_feature in ['angle_of_attack', 'phase', 'geom_dist', 'geom_body_near', 'nearest_particle_distance', 'local_neighbor_count']:
            feature_i = feature_index_or_none(optional_feature)
            if feature_i is not None:
                row[optional_feature] = float(np.median(input_used[:, feature_i]))
        rows.append(row)
    return rows


summary_by_split = {
    split_name: collect_frame_level_summary(frame_ids)
    for split_name, frame_ids in split_frame_ids.items()
}

print('Frame-level summary quantiles')
for split_name, rows in summary_by_split.items():
    if len(rows) == 0:
        print(f'  {split_name}: no frames')
        continue
    particle_counts = np.asarray([row['particle_count'] for row in rows], dtype=np.float64)
    velocity_95 = np.asarray([row['velocity_95'] for row in rows], dtype=np.float64)
    gradient_95 = np.asarray([row['gradient_95'] for row in rows], dtype=np.float64)
    print(
        f"  {split_name}: frames={len(rows)}, "
        f"particles median={np.median(particle_counts):.0f}, "
        f"|u|95 median={np.median(velocity_95):.4g}, "
        f"|gradU|95 median={np.median(gradient_95):.4g}"
    )

figure, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

for split_name, rows in summary_by_split.items():
    if len(rows) == 0:
        continue
    color = split_colors[split_name]
    particle_counts = np.asarray([row['particle_count'] for row in rows], dtype=np.float64)
    velocity_95 = np.asarray([row['velocity_95'] for row in rows], dtype=np.float64)
    gradient_95 = np.asarray([row['gradient_95'] for row in rows], dtype=np.float64)
    frames = np.asarray([row['physical_frame'] for row in rows], dtype=np.float64)

    axes[0, 0].hist(particle_counts, bins=35, histtype='step', linewidth=2.0, color=color, label=split_name)
    axes[0, 1].scatter(frames, velocity_95, s=18, alpha=0.75, color=color, label=split_name)
    axes[1, 0].scatter(frames, gradient_95, s=18, alpha=0.75, color=color, label=split_name)

    sorted_velocity = np.sort(velocity_95)
    cdf = np.linspace(0.0, 1.0, sorted_velocity.size, endpoint=True)
    axes[1, 1].plot(sorted_velocity, cdf, linewidth=2.0, color=color, label=split_name)

axes[0, 0].set_title('Particle count per frame')
axes[0, 0].set_xlabel('particles')
axes[0, 0].set_ylabel('frame count')
axes[0, 0].grid(alpha=0.25)
axes[0, 0].legend(frameon=False)

axes[0, 1].set_title('Velocity target scale over time')
axes[0, 1].set_xlabel('simulation frame')
axes[0, 1].set_ylabel('95th percentile |u|')
axes[0, 1].grid(alpha=0.25)
axes[0, 1].legend(frameon=False)

axes[1, 0].set_title('Velocity-gradient target scale over time')
axes[1, 0].set_xlabel('simulation frame')
axes[1, 0].set_ylabel('95th percentile |gradU|')
axes[1, 0].grid(alpha=0.25)
axes[1, 0].legend(frameon=False)

axes[1, 1].set_title('CDF of high-end velocity target scale')
axes[1, 1].set_xlabel('95th percentile |u| per frame')
axes[1, 1].set_ylabel('cumulative probability')
axes[1, 1].grid(alpha=0.25)
axes[1, 1].legend(frameon=False)

plot_path = results_folder / 'gno_data_pair_distribution_summary.png'
figure.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)

# Optional conditioning/geometry panel: drawn only for features present in the preprocessed file.
optional_panels = [name for name in ['angle_of_attack', 'phase', 'geom_dist', 'nearest_particle_distance', 'local_neighbor_count'] if name in feature_names]
if optional_panels:
    columns = min(3, len(optional_panels))
    rows = int(np.ceil(len(optional_panels) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(5.2 * columns, 3.5 * rows), constrained_layout=True)
    axes = np.asarray(axes).reshape(-1)

    for axis, feature_name in zip(axes, optional_panels):
        for split_name, frame_ids in split_frame_ids.items():
            values = collect_feature_values(frame_ids, feature_name, normalized=False, maximum_frames=50, maximum_particles_per_frame=7000)
            if values is None:
                continue
            axis.hist(values, bins=60, density=True, histtype='step', linewidth=1.9, color=split_colors[split_name], label=split_name)
        axis.set_title(feature_name)
        axis.set_ylabel('density')
        axis.grid(alpha=0.25)

    for axis in axes[len(optional_panels):]:
        axis.axis('off')
    axes[0].legend(frameon=False)
    figure.suptitle('Conditioning and geometry distributions used by the GNO', fontsize=14)
    plot_path = results_folder / 'gno_conditioning_geometry_distributions.png'
    figure.savefig(plot_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)
else:
    print('No conditioning/geometry features found for the optional distribution panel.')


In [ ]:
# Dataset wrapper.
# One item is one simulation frame. The frame contains many particles, so the model sees a graph.

class ParticleFrameDataset(Dataset):
    def __init__(self, frame_ids):
        self.frame_ids = [int(frame_id) for frame_id in frame_ids]

    def __len__(self):
        return len(self.frame_ids)

    def __getitem__(self, index):
        frame_id = self.frame_ids[index]
        input_normalized = torch.from_numpy(inputs_by_frame_normalized[frame_id])
        target_normalized = torch.from_numpy(targets_by_frame_normalized[frame_id])
        input_raw = torch.from_numpy(inputs_by_frame_raw[frame_id])
        context = frame_contexts[frame_id]
        metadata = {
            'frame_id': frame_id,
            'case': str(context.get('case', 'unknown')),
            'frame': str(context.get('frame', context.get('fr', 'unknown'))),
            'particle_count': int(context.get('n_particles', input_normalized.shape[0])),
        }
        return input_normalized, target_normalized, input_raw, metadata


def collate_one_frame(batch):
    # Batch size is one frame. Returning lists keeps variable particle counts simple.
    inputs, targets, raw_inputs, metadata = zip(*batch)
    return list(inputs), list(targets), list(raw_inputs), list(metadata)

training_dataset = ParticleFrameDataset(training_frame_ids)
validation_dataset = ParticleFrameDataset(validation_frame_ids)
testing_dataset = ParticleFrameDataset(testing_frame_ids)
testing_normal_dataset = ParticleFrameDataset(testing_normal_frame_ids)
testing_super_resolution_dataset = ParticleFrameDataset(testing_super_resolution_frame_ids)
testing_unseen_angle_dataset = ParticleFrameDataset(testing_unseen_angle_frame_ids)

training_loader = DataLoader(training_dataset, batch_size=1, shuffle=True, collate_fn=collate_one_frame)
validation_loader = DataLoader(validation_dataset, batch_size=1, shuffle=False, collate_fn=collate_one_frame)
testing_loader = DataLoader(testing_dataset, batch_size=1, shuffle=False, collate_fn=collate_one_frame)
testing_normal_loader = DataLoader(testing_normal_dataset, batch_size=1, shuffle=False, collate_fn=collate_one_frame)
testing_super_resolution_loader = DataLoader(testing_super_resolution_dataset, batch_size=1, shuffle=False, collate_fn=collate_one_frame)
testing_unseen_angle_loader = DataLoader(testing_unseen_angle_dataset, batch_size=1, shuffle=False, collate_fn=collate_one_frame)

sample_input, sample_target, sample_raw, sample_metadata = training_dataset[0]
print('One frame input shape :', tuple(sample_input.shape))
print('One frame target shape:', tuple(sample_target.shape))
print('Example metadata      :', sample_metadata)


In [ ]:
# Small graph neural operator model.
# GNO is appropriate here because the input is a Lagrangian particle cloud, not a regular grid.
# The first three input features are particle coordinates. The graph block exchanges information between nearby particles.

try:
    from neuralop.layers.gno_block import GNOBlock
except Exception as error:
    raise RuntimeError('This notebook needs neuralop with GNOBlock installed in the active environment.') from error


def relative_l2_error(prediction, target, small_number=1e-12):
    difference = (prediction - target).reshape(prediction.shape[0], -1)
    reference = target.reshape(target.shape[0], -1)
    return (torch.linalg.norm(difference, dim=1) / torch.linalg.norm(reference, dim=1).clamp_min(small_number)).mean()


class ParticleVelocityGradientModel(nn.Module):
    def __init__(self, input_size, output_size, hidden_size=96, graph_layers=2, neighbor_radius=0.12, dropout=0.04):
        super().__init__()
        self.neighbor_radius = float(neighbor_radius)

        # Per-particle encoder: maps physical/features channels into a learned latent representation.
        self.input_network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size),
        )

        block_arguments = {}
        signature = inspect.signature(GNOBlock.__init__)
        if 'use_torch_scatter_reduce' in signature.parameters:
            try:
                import torch_scatter  # noqa: F401
                block_arguments['use_torch_scatter_reduce'] = True
            except Exception:
                block_arguments['use_torch_scatter_reduce'] = False
        if 'use_open3d_neighbor_search' in signature.parameters:
            try:
                import open3d  # noqa: F401
                block_arguments['use_open3d_neighbor_search'] = True
            except Exception:
                block_arguments['use_open3d_neighbor_search'] = False

        # Each graph block aggregates neighbor information inside a radius.
        # More layers increase multi-hop information flow but make each epoch slower.
        self.graph_blocks = nn.ModuleList([
            GNOBlock(
                in_channels=hidden_size,
                out_channels=hidden_size,
                coord_dim=3,
                radius=self.neighbor_radius,
                transform_type='linear',
                reduction='mean',
                pos_embedding_type='transformer',
                pos_embedding_channels=16,
                channel_mlp_layers=[hidden_size, hidden_size, hidden_size],
                **block_arguments,
            )
            for _ in range(graph_layers)
        ])
        self.normalization_layers = nn.ModuleList([nn.LayerNorm(hidden_size) for _ in range(graph_layers)])

        self.output_network = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, output_size),
        )

        self.backend_information = block_arguments

    def forward(self, particle_features):
        particle_positions = particle_features[:, :3]
        hidden = self.input_network(particle_features)

        neighbor_cache = None
        for graph_block, normalization_layer in zip(self.graph_blocks, self.normalization_layers):
            if neighbor_cache is None:
                neighbor_cache = graph_block.neighbor_search(
                    data=particle_positions,
                    queries=particle_positions,
                    radius=self.neighbor_radius,
                )
            embedded_positions = graph_block.pos_embedding(particle_positions) if graph_block.pos_embedding is not None else particle_positions
            update = graph_block.integral_transform(
                y=embedded_positions,
                x=embedded_positions,
                neighbors=neighbor_cache,
                f_y=hidden,
            )
            if update.ndim == 3 and update.shape[0] == 1:
                update = update.squeeze(0)
            hidden = normalization_layer(hidden + update)  # residual connection keeps deeper graph stacks trainable

        return self.output_network(hidden)


# Change this for an ablation run. Use 'baseline' first, then 'larger' only if the baseline learns.
model_size = 'larger'  # choices: 'tiny', 'baseline', 'larger'
model_options = {
    'tiny': {'hidden_size': 64, 'graph_layers': 1, 'neighbor_radius': 0.12, 'dropout': 0.04},
    'baseline': {'hidden_size': 96, 'graph_layers': 2, 'neighbor_radius': 0.12, 'dropout': 0.04},
    'larger': {'hidden_size': 128, 'graph_layers': 3, 'neighbor_radius': 0.14, 'dropout': 0.06},
}
model_config = model_options[model_size]

model = ParticleVelocityGradientModel(
    input_size=input_dimension,
    output_size=output_dimension,
    **model_config,
).to(device)

trainable_parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print('Model size:', model_size)
print('Model config:', model_config)
print('Trainable parameters:', f'{trainable_parameter_count:,}')
print('Graph backend info  :', model.backend_information)


In [ ]:
# Training settings.
# Start with quick_check. For final training, switch to better_check.

training_profile = 'quick_check'      # choices: 'quick_check', 'better_check'
training_frame_selection = 'stride'   # choices: 'random_subset', 'all_frames', 'stride'
training_frame_stride = 6             # used only when training_frame_selection='stride'
loss_name = 'smooth_l1'               # choices: 'smooth_l1', 'mse', 'relative_l2'

profiles = {
    'quick_check': {
        'epochs': 12,
        'frames_per_epoch': 12,
        'particles_per_frame_start': 768,
        'particles_per_frame_final': 2048,
        'maximum_particles_before_gpu': 6000,
        'validation_frames_per_check': 8,
        'check_every_epochs': 1,
    },
    'better_check': {
        'epochs': 50,
        'frames_per_epoch': 28,
        'particles_per_frame_start': 1024,
        'particles_per_frame_final': 4096,
        'maximum_particles_before_gpu': 12000,
        'validation_frames_per_check': 16,
        'check_every_epochs': 2,
    },
}
settings = dict(profiles[training_profile])

if training_frame_selection == 'all_frames':
    settings['frames_per_epoch'] = len(training_dataset)
elif training_frame_selection == 'stride':
    # One-in-N frames per epoch. The offset rotates each epoch, so stride=6 sees all frames over six epochs.
    settings['frames_per_epoch'] = int(np.ceil(len(training_dataset) / max(int(training_frame_stride), 1)))
elif training_frame_selection == 'random_subset':
    settings['frames_per_epoch'] = min(settings['frames_per_epoch'], len(training_dataset))
else:
    raise ValueError("training_frame_selection must be 'random_subset', 'all_frames', or 'stride'")

initial_learning_rate = 3e-4
minimum_learning_rate = 5e-6
weight_decay = 3e-5
# Loss balancing between the two physical tasks.
# Default here follows the statistically normalized multi-task loss:
#   L = MSE(u) / std(u_train)^2 + MSE(gradU) / std(gradU_train)^2
# The std values are computed from TRAINING targets only.
loss_balance_mode = 'variance_normalized_physical_mse'
# Other choices: 'equal_task_mean', 'manual'
manual_velocity_loss_weight = 1.0      # used only when loss_balance_mode='manual'
manual_gradient_loss_weight = 1.0      # used only when loss_balance_mode='manual'
smooth_l1_beta = 0.06
gradient_clip_norm = 1.0



def training_task_standard_deviations(frame_ids):
    # Compute scalar task scales from raw TRAINING targets only.
    # This is the exact notebook equivalent of:
    #   u_std = train_targets[:, :3].std()
    #   gradU_std = train_targets[:, 3:].std()
    # but done frame-by-frame to avoid concatenating the whole dataset in memory.
    velocity_count = 0
    velocity_sum = 0.0
    velocity_square_sum = 0.0
    gradient_count = 0
    gradient_sum = 0.0
    gradient_square_sum = 0.0

    for frame_id in np.asarray(frame_ids, dtype=np.int64):
        target = targets_by_frame_raw[int(frame_id)].astype(np.float64, copy=False)
        velocity = target[:, :3].reshape(-1)
        gradient = target[:, 3:].reshape(-1)

        velocity_count += velocity.size
        velocity_sum += float(velocity.sum())
        velocity_square_sum += float(np.dot(velocity, velocity))

        gradient_count += gradient.size
        gradient_sum += float(gradient.sum())
        gradient_square_sum += float(np.dot(gradient, gradient))

    velocity_mean = velocity_sum / max(velocity_count, 1)
    gradient_mean = gradient_sum / max(gradient_count, 1)
    velocity_variance = max(velocity_square_sum / max(velocity_count, 1) - velocity_mean ** 2, 1e-12)
    gradient_variance = max(gradient_square_sum / max(gradient_count, 1) - gradient_mean ** 2, 1e-12)
    return float(np.sqrt(velocity_variance)), float(np.sqrt(gradient_variance))


velocity_task_std_value, gradient_task_std_value = training_task_standard_deviations(training_frame_ids)
velocity_task_std = torch.tensor(velocity_task_std_value, dtype=torch.float32, device=device)
gradient_task_std = torch.tensor(gradient_task_std_value, dtype=torch.float32, device=device)

optimizer = torch.optim.AdamW(model.parameters(), lr=initial_learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=settings['epochs'],
    eta_min=minimum_learning_rate,
)

use_mixed_precision = device.type == 'cuda'
mixed_precision_dtype = torch.bfloat16 if (use_mixed_precision and torch.cuda.is_bf16_supported()) else torch.float16
try:
    scaler = torch.amp.GradScaler('cuda', enabled=bool(use_mixed_precision and mixed_precision_dtype == torch.float16))
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=bool(use_mixed_precision and mixed_precision_dtype == torch.float16))
autocast_options = dict(device_type=device.type, dtype=mixed_precision_dtype, enabled=use_mixed_precision)

if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

print('Training profile:', training_profile)
print('Training frame selection:', training_frame_selection)
print('Training frame stride:', training_frame_stride if training_frame_selection == 'stride' else 'not used')
print('Loss:', loss_name)
print(json.dumps(settings, indent=2))
print('Initial learning rate:', initial_learning_rate)
print('Minimum learning rate:', minimum_learning_rate)
print('Loss balance mode:', loss_balance_mode)
print('Training velocity std:', f'{velocity_task_std_value:.6e}')
print('Training gradU std   :', f'{gradient_task_std_value:.6e}')
if loss_balance_mode == 'manual':
    print('Manual velocity loss weight:', manual_velocity_loss_weight)
    print('Manual velocity-gradient loss weight:', manual_gradient_loss_weight)
elif loss_balance_mode == 'equal_task_mean':
    print('Task weighting: 0.5 * velocity_loss + 0.5 * velocity_gradient_loss')
elif loss_balance_mode == 'variance_normalized_physical_mse':
    print('Task weighting: MSE(u_phys)/u_std^2 + MSE(gradU_phys)/gradU_std^2')
print('Mixed precision:', use_mixed_precision, str(mixed_precision_dtype))


In [ ]:
# Helper functions for sampling, loss, and metrics.
# The model trains on normalized targets. Metrics are reported after converting back to physical units.

# Relative error is undefined when the true target is exactly zero.
# Some frame-0 targets are zero, so we skip those frames for relative-error averages
# and report RMSE/MAE as stable absolute-error diagnostics.
minimum_target_norm_for_relative_error = 1e-6


def denormalize_target(target_normalized):
    return target_normalized * output_standard_deviation + output_mean


def particle_cap_for_epoch(epoch_number):
    if settings['epochs'] <= 1:
        return settings['particles_per_frame_final']
    progress = (epoch_number - 1) / max(settings['epochs'] - 1, 1)
    start = settings['particles_per_frame_start']
    final = settings['particles_per_frame_final']
    return int(round((1.0 - progress) * start + progress * final))


def subsample_before_gpu(input_tensor, target_tensor, raw_input_tensor, maximum_particles):
    # This limits extremely large frames before GPU transfer.
    particle_count = input_tensor.shape[0]
    if particle_count <= maximum_particles:
        return input_tensor, target_tensor, raw_input_tensor
    picked = torch.randperm(particle_count)[:maximum_particles]
    return input_tensor[picked], target_tensor[picked], raw_input_tensor[picked]


def subsample_for_training(input_tensor, target_tensor, raw_input_tensor, particle_cap):
    # This is the within-frame particle curriculum. It keeps quick checks affordable.
    particle_count = input_tensor.shape[0]
    if particle_count <= particle_cap:
        return input_tensor, target_tensor, raw_input_tensor
    picked = torch.randperm(particle_count, device=input_tensor.device)[:particle_cap]
    return input_tensor[picked], target_tensor[picked], raw_input_tensor[picked]


def channel_loss(prediction, target):
    if loss_name == 'mse':
        return (prediction - target) ** 2
    if loss_name == 'relative_l2':
        numerator = (prediction - target) ** 2
        denominator = torch.mean(target ** 2, dim=0, keepdim=True).clamp_min(1e-8)
        return numerator / denominator
    if loss_name == 'smooth_l1':
        return F.smooth_l1_loss(prediction, target, reduction='none', beta=smooth_l1_beta)
    raise ValueError(f'Unknown loss_name: {loss_name}')


def weighted_training_loss(prediction, target):
    # For the variance-normalized option, compute MSE in physical units and
    # divide each task by the variance measured from training targets only.
    # This directly implements:
    #   L = MSE(u) / std(u_train)^2 + MSE(gradU) / std(gradU_train)^2
    if loss_balance_mode == 'variance_normalized_physical_mse':
        prediction_physical = denormalize_target(prediction)
        target_physical = denormalize_target(target)
        velocity_loss = torch.mean((prediction_physical[:, :3] - target_physical[:, :3]) ** 2) / velocity_task_std.clamp_min(1e-12) ** 2
        velocity_gradient_loss = torch.mean((prediction_physical[:, 3:] - target_physical[:, 3:]) ** 2) / gradient_task_std.clamp_min(1e-12) ** 2
        total_loss = velocity_loss + velocity_gradient_loss
        return total_loss, velocity_loss.detach(), velocity_gradient_loss.detach()

    # For normalized-target options, prediction and target are already in standardized target space.
    point_loss = channel_loss(prediction, target)
    velocity_loss = point_loss[:, :3].mean()
    velocity_gradient_loss = point_loss[:, 3:].mean()

    if loss_balance_mode == 'equal_task_mean':
        total_loss = 0.5 * velocity_loss + 0.5 * velocity_gradient_loss
    elif loss_balance_mode == 'manual':
        total_loss = manual_velocity_loss_weight * velocity_loss + manual_gradient_loss_weight * velocity_gradient_loss
    else:
        raise ValueError(f'Unknown loss_balance_mode: {loss_balance_mode}')

    return total_loss, velocity_loss.detach(), velocity_gradient_loss.detach()


def component_metrics(prediction, target):
    # Return absolute metrics and a safe relative L2 value for one tensor block.
    difference = prediction - target
    target_norm = torch.linalg.norm(target.reshape(-1))
    difference_norm = torch.linalg.norm(difference.reshape(-1))
    if target_norm.item() <= minimum_target_norm_for_relative_error:
        relative_error = np.nan
    else:
        relative_error = (difference_norm / target_norm).item()
    rmse = torch.sqrt(torch.mean(difference ** 2)).item()
    mae = torch.mean(torch.abs(difference)).item()
    target_rms = torch.sqrt(torch.mean(target ** 2)).item()
    return relative_error, rmse, mae, target_norm.item(), target_rms


def physical_metrics(prediction_normalized, target_normalized):
    prediction = denormalize_target(prediction_normalized)
    target = denormalize_target(target_normalized)

    full_rel, full_rmse, full_mae, full_target_norm, full_target_rms = component_metrics(prediction, target)
    velocity_rel, velocity_rmse, velocity_mae, velocity_target_norm, velocity_target_rms = component_metrics(
        prediction[:, :3], target[:, :3]
    )
    gradient_rel, gradient_rmse, gradient_mae, gradient_target_norm, gradient_target_rms = component_metrics(
        prediction[:, 3:], target[:, 3:]
    )

    return {
        'relative_error': full_rel,
        'rmse': full_rmse,
        'mae': full_mae,
        'target_norm': full_target_norm,
        'target_rms': full_target_rms,
        'velocity_relative_error': velocity_rel,
        'velocity_rmse': velocity_rmse,
        'velocity_mae': velocity_mae,
        'velocity_target_norm': velocity_target_norm,
        'velocity_target_rms': velocity_target_rms,
        'velocity_gradient_relative_error': gradient_rel,
        'velocity_gradient_rmse': gradient_rmse,
        'velocity_gradient_mae': gradient_mae,
        'velocity_gradient_target_norm': gradient_target_norm,
        'velocity_gradient_target_rms': gradient_target_rms,
    }


def finite_mean(values):
    finite = [float(v) for v in values if np.isfinite(v)]
    return float(np.mean(finite)) if finite else np.nan


@torch.no_grad()
def evaluate_model(data_loader, maximum_frames, particle_cap):
    if len(data_loader.dataset) == 0:
        return {
            'relative_error': np.nan,
            'rmse': np.nan,
            'mae': np.nan,
            'velocity_relative_error': np.nan,
            'velocity_rmse': np.nan,
            'velocity_gradient_relative_error': np.nan,
            'velocity_gradient_rmse': np.nan,
            'frames_evaluated': 0,
            'frames_skipped_for_relative_error': 0,
        }

    model.eval()
    collected = []
    skipped_for_relative = 0

    for frame_index, (inputs, targets, raw_inputs, metadata) in enumerate(data_loader):
        if frame_index >= maximum_frames:
            break
        input_tensor, target_tensor, raw_input_tensor = inputs[0], targets[0], raw_inputs[0]
        input_tensor, target_tensor, raw_input_tensor = subsample_before_gpu(
            input_tensor, target_tensor, raw_input_tensor, settings['maximum_particles_before_gpu']
        )
        input_tensor, target_tensor, raw_input_tensor = subsample_for_training(
            input_tensor.to(device, non_blocking=True),
            target_tensor.to(device, non_blocking=True),
            raw_input_tensor.to(device, non_blocking=True),
            particle_cap,
        )
        with torch.autocast(**autocast_options):
            prediction = model(input_tensor)
        metrics = physical_metrics(prediction.float(), target_tensor.float())
        if not np.isfinite(metrics['relative_error']):
            skipped_for_relative += 1
        collected.append(metrics)

    if not collected:
        return {
            'relative_error': np.nan,
            'rmse': np.nan,
            'mae': np.nan,
            'velocity_relative_error': np.nan,
            'velocity_rmse': np.nan,
            'velocity_gradient_relative_error': np.nan,
            'velocity_gradient_rmse': np.nan,
            'frames_evaluated': 0,
            'frames_skipped_for_relative_error': 0,
        }

    return {
        'relative_error': finite_mean([m['relative_error'] for m in collected]),
        'rmse': finite_mean([m['rmse'] for m in collected]),
        'mae': finite_mean([m['mae'] for m in collected]),
        'velocity_relative_error': finite_mean([m['velocity_relative_error'] for m in collected]),
        'velocity_rmse': finite_mean([m['velocity_rmse'] for m in collected]),
        'velocity_gradient_relative_error': finite_mean([m['velocity_gradient_relative_error'] for m in collected]),
        'velocity_gradient_rmse': finite_mean([m['velocity_gradient_rmse'] for m in collected]),
        'frames_evaluated': int(len(collected)),
        'frames_skipped_for_relative_error': int(skipped_for_relative),
    }


In [ ]:
# Train the model.
# Watch the printed training loss and validation error. For a base check, both should generally move downward.

history = []
best_checkpoint_metric = float('inf')
best_model_state = None
best_epoch = 0
checkpoint_path = results_folder / 'best_task1_v3_model.pt'

In [ ]:
for epoch in range(1, settings['epochs'] + 1):
    epoch_start_time = time.time()
    model.train()
    particle_cap = particle_cap_for_epoch(epoch)

    if training_frame_selection == 'all_frames':
        chosen_training_indices = np.arange(len(training_dataset), dtype=np.int64)
        np.random.default_rng(SEED + epoch).shuffle(chosen_training_indices)
    elif training_frame_selection == 'stride':
        stride = max(int(training_frame_stride), 1)
        offset = (epoch - 1) % stride
        chosen_training_indices = np.arange(offset, len(training_dataset), stride, dtype=np.int64)
        np.random.default_rng(SEED + epoch).shuffle(chosen_training_indices)
    elif training_frame_selection == 'random_subset':
        frame_count_this_epoch = min(settings['frames_per_epoch'], len(training_dataset))
        chosen_training_indices = np.random.choice(len(training_dataset), size=frame_count_this_epoch, replace=False)
    else:
        raise ValueError(f'Unknown training_frame_selection: {training_frame_selection}')

    training_losses = []
    training_velocity_losses = []
    training_gradient_losses = []
    training_relative_errors = []

    for local_step, dataset_index in enumerate(chosen_training_indices, start=1):
        input_tensor, target_tensor, raw_input_tensor, metadata = training_dataset[int(dataset_index)]
        input_tensor, target_tensor, raw_input_tensor = subsample_before_gpu(
            input_tensor, target_tensor, raw_input_tensor, settings['maximum_particles_before_gpu']
        )
        input_tensor = input_tensor.to(device, non_blocking=True)
        target_tensor = target_tensor.to(device, non_blocking=True)
        raw_input_tensor = raw_input_tensor.to(device, non_blocking=True)
        input_tensor, target_tensor, raw_input_tensor = subsample_for_training(
            input_tensor, target_tensor, raw_input_tensor, particle_cap
        )

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(**autocast_options):
            prediction = model(input_tensor)
            loss, velocity_loss, gradient_loss = weighted_training_loss(prediction, target_tensor)

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)
            optimizer.step()

        training_metrics = physical_metrics(prediction.detach().float(), target_tensor.float())
        training_losses.append(float(loss.item()))
        training_velocity_losses.append(float(velocity_loss.item()))
        training_gradient_losses.append(float(gradient_loss.item()))
        training_relative_errors.append(training_metrics['relative_error'])

    scheduler.step()
    current_learning_rate = optimizer.param_groups[0]['lr']

    row = {
        'epoch': epoch,
        'learning_rate': float(current_learning_rate),
        'particle_cap': int(particle_cap),
        'frames_seen': int(len(chosen_training_indices)),
        'frame_selection': training_frame_selection,
        'frame_stride': int(training_frame_stride) if training_frame_selection == 'stride' else 1,
        'frame_stride_offset': int((epoch - 1) % max(int(training_frame_stride), 1)) if training_frame_selection == 'stride' else 0,
        'training_loss': float(np.mean(training_losses)),
        'training_velocity_loss': float(np.mean(training_velocity_losses)),
        'training_velocity_gradient_loss': float(np.mean(training_gradient_losses)),
        'training_relative_error': finite_mean(training_relative_errors),
        'validation_relative_error': np.nan,
        'validation_rmse': np.nan,
        'validation_skipped_zero_frames': 0,
        'testing_relative_error': np.nan,
        'testing_rmse': np.nan,
        'testing_skipped_zero_frames': 0,
        'seconds': float(time.time() - epoch_start_time),
    }

    should_check = (epoch % settings['check_every_epochs'] == 0) or (epoch == settings['epochs'])
    if should_check:
        validation_metrics = evaluate_model(
            validation_loader,
            maximum_frames=settings['validation_frames_per_check'],
            particle_cap=min(particle_cap, settings['particles_per_frame_final']),
        )
        testing_metrics = evaluate_model(
            testing_loader,
            maximum_frames=settings['validation_frames_per_check'],
            particle_cap=min(particle_cap, settings['particles_per_frame_final']),
        )
        row.update({
            'validation_relative_error': validation_metrics['relative_error'],
            'validation_rmse': validation_metrics['rmse'],
            'validation_skipped_zero_frames': validation_metrics['frames_skipped_for_relative_error'],
            'validation_velocity_relative_error': validation_metrics['velocity_relative_error'],
            'validation_velocity_gradient_relative_error': validation_metrics['velocity_gradient_relative_error'],
            'testing_relative_error': testing_metrics['relative_error'],
            'testing_rmse': testing_metrics['rmse'],
            'testing_skipped_zero_frames': testing_metrics['frames_skipped_for_relative_error'],
            'testing_velocity_relative_error': testing_metrics['velocity_relative_error'],
            'testing_velocity_gradient_relative_error': testing_metrics['velocity_gradient_relative_error'],
        })

        # Prefer validation error for checkpointing. If validation data is not generated yet, use training relative error.
        checkpoint_metric = row['validation_relative_error'] if np.isfinite(row['validation_relative_error']) else row['training_relative_error']
        if checkpoint_metric < best_checkpoint_metric:
            best_checkpoint_metric = checkpoint_metric
            best_epoch = epoch
            best_model_state = {name: value.detach().cpu() for name, value in model.state_dict().items()}
            torch.save({
                'model_state_dict': best_model_state,
                'feature_names': feature_names,
                'target_names': target_names,
                'settings': settings,
                'model_config': model_config,
                'loss_name': loss_name,
                'best_epoch': best_epoch,
                'best_checkpoint_metric': best_checkpoint_metric,
            }, checkpoint_path)

    history.append(row)
    print(
        f"epoch {epoch:03d} | "
        f"train loss {row['training_loss']:.4e} | "
        f"train rel {row['training_relative_error']:.4f} | "
        f"val rel {row['validation_relative_error']:.4f} | "
        f"val rmse {row['validation_rmse']:.3e} | "
        f"test rel {row['testing_relative_error']:.4f} | "
        f"test rmse {row['testing_rmse']:.3e} | "
        f"frames {row['frames_seen']} | "
        f"stride {row['frame_stride']} offset {row['frame_stride_offset']} | "
        f"particles {particle_cap} | "
        f"learning rate {current_learning_rate:.2e} | "
        f"skip0 val/test {row['validation_skipped_zero_frames']}/{row['testing_skipped_zero_frames']} | "
        f"seconds {row['seconds']:.1f}",
        flush=True,
    )

if best_model_state is not None:
    model.load_state_dict(best_model_state)

print('Best checkpoint metric:', best_checkpoint_metric)
print('Best epoch:', best_epoch)
print('Checkpoint:', checkpoint_path)


In [ ]:
# Plot the basic learning curves.
# The learning rate has its own plot so it does not look like zero beside particle counts.

if len(history) == 0:
    raise RuntimeError('Run the training cell first.')

epochs = [row['epoch'] for row in history]
training_loss = [row['training_loss'] for row in history]
training_relative_error = [row['training_relative_error'] for row in history]
validation_relative_error = [row['validation_relative_error'] for row in history]
testing_relative_error = [row['testing_relative_error'] for row in history]
learning_rates = [row['learning_rate'] for row in history]
particle_caps = [row['particle_cap'] for row in history]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), constrained_layout=True)

In [ ]:
axes[0].plot(epochs, training_loss, marker='o', label='training loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Normalized training loss')
axes[0].set_title('Training loss')
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(epochs, training_relative_error, marker='o', label='training')
axes[1].plot(epochs, validation_relative_error, marker='o', label='validation')
axes[1].plot(epochs, testing_relative_error, marker='o', label='testing')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Relative error in physical units')
axes[1].set_title('Does the model improve?')
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

axes[2].plot(epochs, learning_rates, marker='o', color='tab:orange', label='learning rate')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning rate')
axes[2].set_title('Learning-rate schedule')
axes[2].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
axes[2].grid(alpha=0.25)
axes[2].legend(frameon=False)

plot_path = results_folder / 'basic_learning_curves.png'
fig.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)

fig, axis = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
axis.plot(epochs, particle_caps, marker='o', color='tab:purple')
axis.set_xlabel('Epoch')
axis.set_ylabel('Particles used from each frame')
axis.set_title('Particle count used during training')
axis.grid(alpha=0.25)
plot_path = results_folder / 'particles_used_per_epoch.png'
fig.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)

In [ ]:
# Final metrics using the best checkpoint loaded at the end of training.
# Empty validation/testing splits are skipped while those cases are still being generated.

final_particle_cap = settings['particles_per_frame_final']
final_metrics = {}
final_metrics['training'] = evaluate_model(training_loader, maximum_frames=12, particle_cap=final_particle_cap)
if len(validation_dataset) > 0:
    final_metrics['validation'] = evaluate_model(validation_loader, maximum_frames=20, particle_cap=final_particle_cap)
if len(testing_dataset) > 0:
    final_metrics['testing_all'] = evaluate_model(testing_loader, maximum_frames=20, particle_cap=final_particle_cap)
if len(testing_normal_dataset) > 0:
    final_metrics['testing_normal'] = evaluate_model(testing_normal_loader, maximum_frames=20, particle_cap=final_particle_cap)
if len(testing_super_resolution_dataset) > 0:
    final_metrics['testing_super_resolution'] = evaluate_model(testing_super_resolution_loader, maximum_frames=20, particle_cap=final_particle_cap)
if len(testing_unseen_angle_dataset) > 0:
    final_metrics['testing_unseen_angle'] = evaluate_model(testing_unseen_angle_loader, maximum_frames=20, particle_cap=final_particle_cap)

print(json.dumps(final_metrics, indent=2))

figure, axis = plt.subplots(figsize=(7.2, 4.5), constrained_layout=True)
split_names = list(final_metrics.keys())
values = [final_metrics[name]['relative_error'] for name in split_names]
metric_colors = ['#4c78a8', '#59a14f', '#e15759', '#b07aa1', '#f28e2b']
bars = axis.bar(split_names, values, color=metric_colors[:len(split_names)])
axis.tick_params(axis='x', rotation=20)
axis.set_ylabel('Relative error in physical units')
axis.set_title('Final error by split')
axis.grid(axis='y', alpha=0.25)
for bar, value in zip(bars, values):
    axis.text(bar.get_x() + bar.get_width() / 2, value, f'{value:.3f}', ha='center', va='bottom')
plot_path = results_folder / 'final_relative_error_by_split.png'
figure.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)


In [ ]:
# Prediction-versus-true scatter plots.
# These are often more revealing than loss alone: diagonal alignment means good magnitude prediction.

@torch.no_grad()
def collect_prediction_samples(dataset, maximum_frames=6, maximum_particles_per_frame=5000):
    if len(dataset) == 0:
        return None, None
    true_chunks = []
    predicted_chunks = []

    for dataset_index in range(min(len(dataset), maximum_frames)):
        input_tensor, target_tensor, raw_input_tensor, metadata = dataset[dataset_index]
        input_tensor, target_tensor, raw_input_tensor = subsample_before_gpu(
            input_tensor, target_tensor, raw_input_tensor, maximum_particles_per_frame
        )
        prediction_normalized = model(input_tensor.to(device)).float()
        true_physical = denormalize_target(target_tensor.to(device)).cpu().numpy()
        predicted_physical = denormalize_target(prediction_normalized).cpu().numpy()
        true_chunks.append(true_physical)
        predicted_chunks.append(predicted_physical)

    return np.concatenate(true_chunks, axis=0), np.concatenate(predicted_chunks, axis=0)


def plot_scatter_for_split(split_name, dataset):
    true_values, predicted_values = collect_prediction_samples(dataset)
    if true_values is None:
        print(f'[{split_name}] no data available; skipping scatter plot.')
        return

    true_velocity = np.linalg.norm(true_values[:, :3], axis=1)
    predicted_velocity = np.linalg.norm(predicted_values[:, :3], axis=1)
    true_gradient = np.linalg.norm(true_values[:, 3:], axis=1)
    predicted_gradient = np.linalg.norm(predicted_values[:, 3:], axis=1)

    figure, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), constrained_layout=True)
    for axis, true_array, predicted_array, title in [
        (axes[0], true_velocity, predicted_velocity, '|u|'),
        (axes[1], true_gradient, predicted_gradient, '|gradU|'),
    ]:
        if true_array.shape[0] > 30000:
            picked = np.random.default_rng(SEED).choice(true_array.shape[0], size=30000, replace=False)
            true_array = true_array[picked]
            predicted_array = predicted_array[picked]
        low = float(np.quantile(np.concatenate([true_array, predicted_array]), 0.01))
        high = float(np.quantile(np.concatenate([true_array, predicted_array]), 0.99))
        axis.scatter(true_array, predicted_array, s=3, alpha=0.25)
        axis.plot([low, high], [low, high], color='black', linewidth=1.2, linestyle='--')
        axis.set_xlabel(f'true {title}')
        axis.set_ylabel(f'predicted {title}')
        axis.set_title(f'{split_name}: {title}')
        axis.grid(alpha=0.25)

    plot_path = results_folder / f'{split_name.lower()}_prediction_scatter.png'
    figure.savefig(plot_path, dpi=220, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


plot_scatter_for_split('training', training_dataset)
plot_scatter_for_split('validation', validation_dataset)
plot_scatter_for_split('testing_all', testing_dataset)
plot_scatter_for_split('testing_normal', testing_normal_dataset)
plot_scatter_for_split('testing_super_resolution', testing_super_resolution_dataset)
plot_scatter_for_split('testing_unseen_angle', testing_unseen_angle_dataset)


In [ ]:
# Prediction plots for training, validation, and testing.
# Coordinates are input particle locations. The model predicts velocity and velocity gradient values on those particles.

@torch.no_grad()
def predict_one_frame(dataset, dataset_index=0, maximum_particles_for_plot=12000):
    model.eval()
    input_tensor, target_tensor, raw_input_tensor, metadata = dataset[dataset_index]
    input_tensor, target_tensor, raw_input_tensor = subsample_before_gpu(
        input_tensor, target_tensor, raw_input_tensor, maximum_particles_for_plot
    )
    prediction_normalized = model(input_tensor.to(device)).float()
    target_physical = denormalize_target(target_tensor.to(device)).cpu().numpy()
    prediction_physical = denormalize_target(prediction_normalized).cpu().numpy()
    raw_input = raw_input_tensor.numpy()
    return raw_input[:, :3], target_physical, prediction_physical, metadata


def robust_color_limits(values, lower=0.02, upper=0.98):
    finite_values = np.asarray(values)[np.isfinite(values)]
    if finite_values.size == 0:
        return 0.0, 1.0
    low = float(np.quantile(finite_values, lower))
    high = float(np.quantile(finite_values, upper))
    if low >= high:
        high = low + 1e-6
    return low, high


def plot_prediction_example(split_name, dataset, dataset_index=0):
    coordinates, true_values, predicted_values, metadata = predict_one_frame(dataset, dataset_index=dataset_index)

    true_velocity = np.linalg.norm(true_values[:, :3], axis=1)
    predicted_velocity = np.linalg.norm(predicted_values[:, :3], axis=1)
    velocity_error = predicted_velocity - true_velocity

    true_gradient = np.linalg.norm(true_values[:, 3:], axis=1)
    predicted_gradient = np.linalg.norm(predicted_values[:, 3:], axis=1)
    gradient_error = predicted_gradient - true_gradient

    figure, axes = plt.subplots(2, 3, figsize=(15, 8.5), constrained_layout=True)

    velocity_low, velocity_high = robust_color_limits(np.concatenate([true_velocity, predicted_velocity]))
    gradient_low, gradient_high = robust_color_limits(np.concatenate([true_gradient, predicted_gradient]))
    velocity_error_limit = max(abs(robust_color_limits(velocity_error)[0]), abs(robust_color_limits(velocity_error)[1]), 1e-12)
    gradient_error_limit = max(abs(robust_color_limits(gradient_error)[0]), abs(robust_color_limits(gradient_error)[1]), 1e-12)

    plot_items = [
        (0, 0, true_velocity, '|u| true', 'viridis', velocity_low, velocity_high),
        (0, 1, predicted_velocity, '|u| prediction', 'viridis', velocity_low, velocity_high),
        (0, 2, velocity_error, '|u| signed error', 'RdBu_r', -velocity_error_limit, velocity_error_limit),
        (1, 0, true_gradient, '|gradU| true', 'magma', gradient_low, gradient_high),
        (1, 1, predicted_gradient, '|gradU| prediction', 'magma', gradient_low, gradient_high),
        (1, 2, gradient_error, '|gradU| signed error', 'RdBu_r', -gradient_error_limit, gradient_error_limit),
    ]

    for row, column, values, title, color_map, low, high in plot_items:
        scatter = axes[row, column].scatter(
            coordinates[:, 0], coordinates[:, 2], c=values, s=3, cmap=color_map, vmin=low, vmax=high
        )
        axes[row, column].set_title(title)
        axes[row, column].set_xlabel('x')
        axes[row, column].set_ylabel('z')
        axes[row, column].grid(alpha=0.18)
        plt.colorbar(scatter, ax=axes[row, column], fraction=0.046, pad=0.02)

    figure.suptitle(
        f"{split_name}: case {metadata['case']}, frame {metadata['frame']}, particles shown {coordinates.shape[0]}",
        fontsize=13,
    )
    plot_path = results_folder / f'{split_name.lower()}_prediction_example.png'
    figure.savefig(plot_path, dpi=240, bbox_inches='tight')
    plt.show()
    print('Saved:', plot_path)


plot_prediction_example('training', training_dataset, dataset_index=0)
if len(validation_dataset) > 0:
    plot_prediction_example('validation', validation_dataset, dataset_index=0)
else:
    print('[validation] no data available; skipping prediction image.')
for split_name, split_dataset in [
    ('testing_all', testing_dataset),
    ('testing_normal', testing_normal_dataset),
    ('testing_super_resolution', testing_super_resolution_dataset),
    ('testing_unseen_angle', testing_unseen_angle_dataset),
]:
    if len(split_dataset) > 0:
        plot_prediction_example(split_name, split_dataset, dataset_index=0)
    else:
        print(f'[{split_name}] no data available; skipping prediction image.')
